%matplotlib inline

In [1]:
import os
os.environ['USE_PYGEOS'] = '0'
os.environ['PROJ_DEBUG'] = '2' # For showing logs
#os.environ['GDAL_IGNORE_COG_LAYOUT_BREAK'] = 'YES'
#os.environ['PROJ_NETWORK'] = 'ON' # Ensure this is 'ON' to get shift grids over the internet
print(os.environ['PROJ_NETWORK']) 
 
import geopandas as gpd
import shapely
import rasterio
import rioxarray
import matplotlib.pyplot as plt
#https://xyzservices.readthedocs.io/en/stable/introduction.html
import xyzservices
#from fetch_dem import opentopo_utils
#import easysnowdata
import numpy as np
import coincident

ON


/panfs/ccds02/nobackup/people/sbhusha1/sw/coincident/src/coincident/search/main.py:14: UserWarning: Unable to authenticate with Maxar API. Please set MAXAR_API_KEY environment variable.
  from coincident.search import neon_api, opentopo_api, stac, wesm


In [3]:
def fetch_worldcover(raster_fn,match_grid_da=None):
    with rasterio.open(raster_fn) as dataset:
        bounds = dataset.bounds
        bounds = rasterio.warp.transform_bounds(dataset.crs, 'EPSG:4326', *bounds)
        bbox_gdf = gpd.GeoDataFrame(geometry=[shapely.box(*bounds)],crs='EPSG:4326',index=[0])
        
    
    # Load Worldcover DEM for this bounding box (from Scott's notebook)
    gf_wc = coincident.search.search(
        dataset="worldcover",
        intersects=bbox_gdf,
    )  # Asset of interest = 'data'
    if match_grid_da is not None:
        ds_wc = coincident.io.xarray.to_dataset(
            gf_wc,
            bands=["map"],
            #aoi=bbox,
            like=match_grid_da, # Match grid of COP30 (both considered 4326)
        )
    else:
        ds_wc = coincident.io.xarray.to_dataset(
            gf_wc,
            bands=["map"],
            #aoi=bbox,
            )
    da_wc = ds_wc.isel(time=1).to_dataarray().squeeze()
    return da_wc


def common_mask(da_list,apply=False):
    """
    From a list of xarray dataarray objects sharing the same projection/extent/res, compute common mask where all input datasets have non-nan pixels
    """
    # load nan layers as numpy array
    nan_arrays = np.array([np.isnan(da.values) for da in da_list])
    common_mask = 1 - np.any(nan_arrays,axis=0)
    
    if apply:
        common_mask_da_list = [da.where(common_mask,np.nan) for da in da_list]
        return common_mask_da_list
    else:
        return common_mask

def fetch_cop30(raster_fn,match_grid_da=None):
    with rasterio.open(raster_fn) as dataset:
        bounds = dataset.bounds
        bounds = rasterio.warp.transform_bounds(dataset.crs, 'EPSG:4326', *bounds)
        bbox_gdf = gpd.GeoDataFrame(geometry=[shapely.box(*bounds)],crs='EPSG:4326',index=[0])
        
    
    # Load Worldcover DEM for this bounding box (from Scott's notebook)
    gf = coincident.search.search(
        dataset="cop30",
        intersects=bbox_gdf,
    )  # Asset of interest = 'data'
    if match_grid_da is not None:
        ds = coincident.io.xarray.to_dataset(
            gf,
            bands=["data"],
            #aoi=bbox,
            like=match_grid_da, # Match grid of COP30 (both considered 4326)
        )
    else:
        ds = coincident.io.xarray.to_dataset(
            gf,
            bands=["data"],
            #aoi=bbox,
            )
    da = ds.to_dataarray().squeeze()
    return da


def confirm_3dep_vertical(raster_fn,bare_diff_tolerance=3):
    lidar_da = rioxarray.open_rasterio(raster_fn,masked=True).squeeze()
    worldcover_da = fetch_worldcover(raster_fn,lidar_da)
    cop30_da = fetch_cop30(raster_fn,lidar_da)
    lidar_da_masked,worldcover_da_masked,cop30_da_masked = common_mask([lidar_da,worldcover_da,cop30_da],apply=True)
    dem_diff = lidar_da_masked - cop30_da_masked
    ## Mask out bare and sparse vegetation class
    bare_sparse_mask = worldcover_da_masked == 60
    dem_diff_bare = dem_diff.where(bare_sparse_mask,np.nan)
    median_diff = np.nanmedian(dem_diff_bare.values)
    print(f"Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is {median_diff:.2f} m")
    if np.abs(median_diff) <= bare_diff_tolerance:
        #this means that both COP30 and 3DEP LiDAR DSM are with respect to geoid
        print("Looks like the 3DEP height estimates are with respect to geiod, will apply vertical datum shift to return heights with respect to ellipsoid")
    else:
        #this means that 3DEP LiDAR DSM is with respect to ellipsoid
        print("Looks like the 3DEP height estimates are already with respect to ellipsoid, geoid to ellipsoid transformation will not be attempted")

In [27]:
%cd /panfs/ccds02/nobackup/people/sbhusha1/pcd/seattle

/panfs/ccds02/nobackup/people/sbhusha1/pcd/seattle


In [28]:
#seattle King-county test
raster_fn = 'seattle_3dep/King-county-21-lidar-DSM_mos.tif'

In [29]:
%time confirm_3dep_vertical(raster_fn)

Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is 2.17 m
Looks like the 3DEP height estimates are with respect to geiod, will apply vertical datum shift to return heights with respect to ellipsoid
CPU times: user 1.74 s, sys: 113 ms, total: 1.85 s
Wall time: 4.66 s


In [30]:
#Mt Baker lidar data test
raster_fn = '../mt-baker/baker_3dep/baker-DSM_mos.tif'

In [31]:
%time confirm_3dep_vertical(raster_fn)

Observed difference between COP30 EGM2008 and 3DEP LiDAR DSM over bareground and sparse vegetation surfaces is -18.24 m
Looks like the 3DEP height estimates are already with respect to ellipsoid, geoid to ellipsoid transformation will not be attempted
CPU times: user 869 ms, sys: 17 ms, total: 886 ms
Wall time: 2.19 s
